# Insurance Claims Review Prioritisation
**Business Management × Data Science**

Synthetic review flags are **not confirmed fraud**. This notebook explains the business decision and loads reproducible outputs. Run `python src/run_all.py` from the repository root before opening.

In [1]:
from pathlib import Path
import pandas as pd
import json
ROOT = Path.cwd()
if not (ROOT/'reports/project_summary.json').exists():
    ROOT = ROOT.parent
assert (ROOT/'reports/project_summary.json').exists(), 'Run the pipeline from the repository root first.'
summary = json.loads((ROOT/'reports/project_summary.json').read_text())
summary

{'dataset': 'Synthetic insurance claims dataset',
 'records': 6000,
 'flagged_claims': 287,
 'synthetic_review_flag_rate': 0.0478,
 'best_model_by_cv_roc_auc': 'Logistic Regression',
 'holdout_roc_auc': 0.5853,
 'holdout_average_precision': 0.0633,
 'cv_roc_auc_mean_std': [0.6168, 0.0381],
 'cv_average_precision_vs_baseline': [0.0853, 0.0478],
 'flags_found_in_top_20pct_of_queue': 0.3043,
 'permutation_test_p_value': 0.0323,
 'purpose': 'Synthetic review-flag prioritisation only; not a fraud-detection or production claim.'}

## 1. The business decision
Claims managers have limited investigation capacity. Our question is **which claims to review**, not whether a customer committed fraud. The review flag is synthetic and is not a verified outcome.

In [2]:
claims = pd.read_csv(ROOT/'data/processed/insurance_claims_clean.csv')
print('Claims:', len(claims), 'Review flags:', int(claims.fraud_review_flag.sum()))
claims[['claim_amount','days_to_report','fraud_review_flag']].describe()

Claims: 6000 Review flags: 287


,claim_amount,days_to_report,fraud_review_flag
count,5910.000000,6000.000000,6000.000000
mean,37431.556968,14.986500,0.047833
std,31911.741334,9.008876,0.213431
min,973.670000,0.000000,0.000000
25%,17105.825000,7.000000,0.000000
50%,28465.055000,15.000000,0.000000
75%,47638.492500,23.000000,0.000000
max,628367.880000,30.000000,1.000000


## 2. Independent model evaluation
Models were selected with five-fold cross-validation on the training split. Only the selected model was evaluated on the hold-out set. Modest ROC-AUC and average precision limit the practical usefulness.

In [3]:
pd.read_csv(ROOT/'reports/cross_validation.csv')

,model,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std,flags_found_in_top_20pct,baseline_average_precision
0,Logistic Regression,0.616801,0.038112,0.085337,0.013049,0.304348,0.047917
1,Random Forest,0.614832,0.030616,0.083250,0.012662,0.330435,0.047917
2,Decision Tree,0.569325,0.020371,0.059604,0.006503,0.269565,0.047917


In [4]:
pd.read_csv(ROOT/'reports/model_comparison.csv')

,model,roc_auc,average_precision,flags_found_in_top_20pct,test_claims
0,Logistic Regression,0.585348,0.063325,0.298246,1200


## 3. Workload prioritisation
The following lift results use full-portfolio out-of-fold scores for **descriptive exploration**. They are not a second independent test.

In [5]:
pd.read_csv(ROOT/'reports/lift_table.csv')

,queue_share,claims_reviewed,flags_found,share_of_flags_found,precision,lift_vs_random,share_of_flagged_claim_value
0,0.02,120,10,0.034843,0.083333,1.742160,0.067838
1,0.05,300,25,0.087108,0.083333,1.742160,0.140517
2,0.10,600,49,0.170732,0.081667,1.707317,0.251387
3,0.20,1200,91,0.317073,0.075833,1.585366,0.448878
4,0.30,1800,118,0.411150,0.065556,1.370499,0.576620
5,0.50,3000,180,0.627178,0.060000,1.254355,0.798435
6,1.00,6000,287,1.000000,0.047833,1.000000,1.000000


## 4. Business impact — hypothetical only
The scenario assumes a review cost and a recovery percentage on synthetic flagged claims. Neither is validated. The optimal queue share was chosen on the same portfolio, so this is not a prospective ROI estimate.

In [6]:
pd.read_csv(ROOT/'reports/review_economics.csv')

,strategy,best_queue_share,claims_reviewed,hypothetical_net_benefit_R
0,Random review,0.01,60,-28347.96675
1,Ranked by model score,0.12,720,281798.43500
2,Ranked by expected recoverable value,0.20,1200,616824.54500


In [7]:
pd.read_csv(ROOT/'reports/review_economics_sensitivity.csv').head(12)

,cost_per_review_R,recovery_share,strategy,best_queue_share,hypothetical_net_benefit_R,pays_off
0,500,0.15,Random review,0.01,0.000000e+00,False
1,500,0.15,Ranked by model score,0.24,3.006432e+05,True
2,500,0.15,Ranked by expected recoverable value,0.21,4.934366e+05,True
3,500,0.25,Random review,1.00,1.652033e+05,True
4,500,0.25,Ranked by model score,0.41,1.079902e+06,True
5,500,0.25,Ranked by expected recoverable value,0.34,1.365335e+06,True
6,500,0.35,Random review,1.00,1.431285e+06,True
7,500,0.35,Ranked by model score,0.52,2.091443e+06,True
8,500,0.35,Ranked by expected recoverable value,0.38,2.334517e+06,True
9,1000,0.15,Random review,0.01,0.000000e+00,False


## 5. Responsible recommendations
Do not automate claim denial or fraud allegations. Before any operational use, collect verified investigation outcomes, test on later claims, validate fairness, establish human escalation and appeal rights, and measure actual costs.